# Binary Classification with a Bank Dataset

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/playground-series-s5e8/train.csv", index_col=0)
test_data = pd.read_csv("/kaggle/input/playground-series-s5e8/test.csv", index_col=0)

Description of each Column 



1. age: Age of the client (numeric)
2. job: Type of job (categorical: "admin.", "blue-collar", "entrepreneur", etc.)
3. marital: Marital status (categorical: "married", "single", "divorced")
4. education: Level of education (categorical: "primary", "secondary", "tertiary", "unknown")
5. default: Has credit in default? (categorical: "yes", "no")
6. balance: Average yearly balance in euros (numeric)
7. housing: Has a housing loan? (categorical: "yes", "no")
8. loan: Has a personal loan? (categorical: "yes", "no")
9. contact: Type of communication contact (categorical: "unknown", "telephone", "cellular")
10. day: Last contact day of the month (numeric, 1-31)
11. month: Last contact month of the year (categorical: "jan", "feb", "mar", …, "dec")
12. duration: Last contact duration in seconds (numeric)
13. campaign: Number of contacts performed during this campaign (numeric)
14. pdays: Number of days since the client was last contacted from a previous campaign (numeric; -1 means the client was not previously contacted)
15. previous: Number of contacts performed before this campaign (numeric)
16. poutcome: Outcome of the previous marketing campaign (categorical: "unknown", "other", "failure", "success")
17. y: The target variable, whether the client subscribed to a term deposit (binary: "yes", "no")

In [ ]:
train_data.head(20)

In [ ]:
train_data.tail(20)

In [ ]:
test_data.head(20)

In [ ]:
train_data['y'].value_counts()

In [ ]:
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
train_data.describe(include="object")

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data.hist(figsize=(20, 15), bins=50, xlabelsize=8, ylabelsize=8)

In [ ]:
sns.countplot(data=train_data, x='marital')

In [ ]:
plt.figure(figsize=(20, 12))
plt.title("Categorical Feature Representation")
plot_count = 0
plt.axis("off")

categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

for category in categorical_cols:
    ax = plt.subplot(3, 3, plot_count+1)
    sns.countplot(data=train_data, x=category)
    plot_count += 1

plt.show()
    

In [ ]:
import random
from sklearn.model_selection import train_test_split

def prepare_data():
    X = train_data.drop(columns='y')
    y = train_data['y'].astype(int)

    train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

    return [train_X, valid_X, train_y, valid_y]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

train_X, valid_X, train_y, valid_y = prepare_data()

numeric_features = train_X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = train_X.select_dtypes(include=['object', 'category']).columns

numeric_transform = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])

categorical_transform = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers= [("num", numeric_transform, numeric_features), ("cat", categorical_transform, categorical_features)])


In [ ]:
from sklearn.linear_model import LogisticRegression


clf = Pipeline(steps=[("preprocessor", preprocessor), ("classifier",  LogisticRegression(random_state = 42, class_weight = "balanced"))])

clf.fit(train_X, train_y)

In [ ]:
predictions = clf.predict(train_X)

In [ ]:
from sklearn.metrics import mean_absolute_error

print("Training Accuracy: ", (predictions == train_y.values).mean())

In [ ]:
validation_prediction = clf.predict(valid_X)

In [ ]:
print("Validation Accuracy: ", (validation_prediction == valid_y.values).mean())

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, roc_curve, auc

print("Train ROC", roc_auc_score(train_y, clf.predict_proba(train_X)[:, 1]))

print("Validation ROC", roc_auc_score(valid_y, clf.predict_proba(valid_X)[:, 1]))

In [ ]:
validation_prob = clf.predict_proba(valid_X)[:, 1]

#Computing ROC curve and AUC
fpr, tpr, thresholds = roc_curve(valid_y, validation_prob)
roc_auc = auc(fpr, tpr)

# Plot ROC Curve
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.utils.class_weight import compute_sample_weight


In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"), 
    "Gradient Boosting": GradientBoostingClassifier(),
    # "Support Vector Machine": SVC(probability = True, random_state=42, class_weight="balanced"),
    "Naive Bayes": GaussianNB()
}

In [ ]:
#Model comparison

result = {}

for name, model in models.items():
    clf = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])

    if name in ["Gradient Boosting", "Naive Bayes"]:
        sample_weights = compute_sample_weight(class_weight="balanced", y = train_y)
        clf.fit(train_X, train_y, classifier__sample_weight=sample_weights if name=="Gradient Boosting" else None)
    else:
        clf.fit(train_X, train_y)

    y_pred_proba = clf.predict_proba(valid_X)[:, 1]  # probs for positive class
    auc = roc_auc_score(valid_y, y_pred_proba)
    result[name] = auc

results_df = pd.DataFrame.from_dict(result, orient="index", columns=["ROC-AUC"])
results_df = results_df.sort_values(by="ROC-AUC", ascending=False)

In [ ]:
results_df

In [ ]:
best_model = results_df['ROC-AUC'].idxmax()

In [ ]:
best_model

### The best model among all the experiments is Random Forest Model
#### Fine-tuning the random forest model




In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from scipy.stats import randint, uniform
from joblib import parallel_backend

In [ ]:
rf_base_model = Pipeline([("proc", preprocessor), ("clf", RandomForestClassifier(random_state=42, n_jobs=2))])

model_params = {
    "clf__n_estimators": randint(100, 400),   
    "clf__max_depth": [None, 10, 20, 30],
    "clf__max_features": ["sqrt", 0.2],
    "clf__min_samples_split": randint(2, 15),
    "clf__min_samples_leaf": randint(1, 8),
    "clf__bootstrap": [True],
    "clf__class_weight": [None, "balanced"],
    "clf__ccp_alpha": [0.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rnd = RandomizedSearchCV(
    rf_base_model,
    param_distributions=model_params,
    n_iter=10,              
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=2,              
    verbose=1
)



#### Subsampling the training data for fine tuning

In [ ]:
X_sub, _, y_sub, _ = train_test_split(
    train_X, train_y,
    stratify=train_y,
    train_size=0.3,
    random_state=42
)


In [ ]:
with parallel_backend('threading', n_jobs=2):
    rnd.fit(X_sub, y_sub)


In [ ]:
best_rf = rnd.best_estimator_


In [ ]:
print("Coarse best (CV AUC):", rnd.best_score_, rnd.best_params_)

In [ ]:
best_rf.set_params(clf__n_estimators=1000)
best_rf.fit(train_X, train_y)

### Extacting feature importance from the Best RF Model
### This helps in understanding which features are driving the predictions

In [ ]:
importances = best_rf.named_steps['clf'].feature_importances_
feature_names = best_rf.named_steps['proc'].get_feature_names_out()
fi = pd.DataFrame({"feature": feature_names, "importance": importances})
fi.sort_values(by="importance", ascending=False).head(20).plot.barh(x="feature", y="importance")
plt.show()

In [ ]:
train_prediction = best_rf.predict(train_X)
train_probs = best_rf.predict_proba(train_X)[:, 1] 

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

In [ ]:
train_auc = roc_auc_score(train_y, train_probs)
train_acc = accuracy_score(train_y, train_prediction)

In [ ]:
print("Train ROC-AUC:", round(train_auc, 4))
print("Train Accuracy:", round(train_acc, 4))

In [ ]:
valid_preds = best_rf.predict(valid_X)
valid_probs = best_rf.predict_proba(valid_X)[:, 1]

In [ ]:
from sklearn import metrics
#Computing ROC curve and AUC
fpr, tpr, thresholds = roc_curve(valid_y, valid_probs)
roc_auc_rf = metrics.auc(fpr, tpr)

# Plot ROC Curve
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f"ROC curve (AUC = {roc_auc_rf:.3f})")
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
valid_auc = roc_auc_score(valid_y, valid_probs)
valid_acc = accuracy_score(valid_y, valid_preds)

In [ ]:
print("\nValidation ROC-AUC:", round(valid_auc, 4))
print("Validation Accuracy:", round(valid_acc, 4))

In [ ]:
print("\nClassification Report (Validation):")
print(classification_report(valid_y, valid_preds))

### Preparing the Submission 

In [ ]:
test_predictions = best_rf.predict(test_data)
test_pred_prob = best_rf.predict_proba(test_data)[:, 1]

In [ ]:
test_predictions

In [ ]:
test_pred_prob

In [ ]:
submission_df = pd.DataFrame({'id': test_data.index, 'y': test_pred_prob})

In [ ]:
submission_df.to_csv("submission.csv", index=False)